In [1]:

## importing required libraries
import os
import shutil
import random
from tqdm.notebook import tqdm

In [2]:
train_path_img = "./yolo_data/images/train/"
train_path_label = "./yolo_data/labels/train/"
val_path_img = "./yolo_data/images/val/"
val_path_label = "./yolo_data/labels/val/"
test_path = "./yolo_data/test"

In [3]:
# !pip install jupyter
# !pip install ipywidgets widgetsnbextension pandas-profiling
# !jupyter nbextension enable --py widgetsnbextension

In [4]:

'''
Split the dataset into train and test and creates the train.txt and test.tx with
the respective path of the images in each folder
'''

def train_test_split(path,neg_path=None, split = 0.2):
    print("------ PROCESS STARTED -------")


    files = list(set([name[:-4] for name in os.listdir(path)])) ## removing duplicate names i.e. counting only number of images


    print (f"--- This folder has a total number of {len(files)} images---")
    random.seed(42)
    random.shuffle(files)

    test_size = int(len(files) * split)
    train_size = len(files) - test_size

    ## creating required directories

    os.makedirs(train_path_img, exist_ok = True)
    os.makedirs(train_path_label, exist_ok = True)
    os.makedirs(val_path_img, exist_ok = True)
    os.makedirs(val_path_label, exist_ok = True)


    ### ----------- copying images to train folder
    for filex in tqdm(files[:train_size]):
      if filex == 'classes':
          continue  
      file_tif = path + filex + '.tif'
      file_txt = path + filex + '.txt'
        
      if not os.path.exists(file_tif):
          print(f"Error: File {file_tif} not found.")
      if not os.path.exists(file_txt):
          print(f"Error: File {file_txt} not found.")
          
      shutil.copy2(path + filex + '.tif',f"{train_path_img}/" + filex + '.tif' )
      shutil.copy2(path + filex + '.txt', f"{train_path_label}/" + filex + '.txt')



    print(f"------ Training data created with 80% split {len(files[:train_size])} images -------")

    if neg_path:
        neg_images = list(set([name[:-4] for name in os.listdir(neg_path)])) ## removing duplicate names i.e. counting only number of images
        for filex in tqdm(neg_images):
            shutil.copy2(neg_path+filex+ ".tif", f"{train_path_img}/" + filex + '.tif')

        print(f"------ Total  {len(neg_images)} negative images added to the training data -------")

        print(f"------ TOTAL Training data created with {len(files[:train_size]) + len(neg_images)} images -------")



    ### copytin images to validation folder
    for filex in tqdm(files[train_size:]):
      if filex == 'classes':
          continue
      # print("running")
      shutil.copy2(path + filex + '.tif', f"{val_path_img}/" + filex + '.tif' )
      shutil.copy2(path + filex + '.txt', f"{val_path_label}/" + filex + '.txt')

    print(f"------ Testing data created with a total of {len(files[train_size:])} images ----------")

    print("------ TASK COMPLETED -------")

## spliting the data into train-test and creating train.txt and test.txt files
# train_test_split('/content/drive/MyDrive/custom_notebooks/yolo_data/')

### for label_tag
train_test_split('./Data/') ### without negative images
# train_test_split('./data/','./negative_images/') ### if you want to feed negative images

------ PROCESS STARTED -------
--- This folder has a total number of 522 images---


  0%|          | 0/418 [00:00<?, ?it/s]

------ Training data created with 80% split 418 images -------


  0%|          | 0/104 [00:00<?, ?it/s]

------ Testing data created with a total of 104 images ----------
------ TASK COMPLETED -------


In [5]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.169 🚀 Python-3.8.20 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3080 Ti, 11922MiB)
Setup complete ✅ (20 CPUs, 15.4 GB RAM, 184.5/287.3 GB disk)


In [6]:
from ultralytics import RTDETR

# Load a model
model = RTDETR("rtdetr-l.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(task="detect" ,data="./dataset_custom.yaml", epochs=200, imgsz=1280, batch=8, project="./Training_Results", name="RTDETR_200_1280", device=0)

Ultralytics 8.3.169 🚀 Python-3.8.20 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3080 Ti, 11922MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset_custom.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=RTDETR_200_12807, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, p

ModuleNotFoundError: No module named 'numpy._core'

In [7]:
import torch
torch.cuda.empty_cache()

In [ ]:
from ultralytics import RTDETR

model = RTDETR("F:/Model_Training/Training_Results/RTDETR_1000_1024/weights/best.pt")

results = model(source="F:/Model_Training/Test_Images",save=True, conf=0.1)

In [ ]:
result = model.val()
print(result.box.maps)

In [ ]:
model = RTDETR("F:/Model_Training/Training_Results/RTDETR_1000_1024/weights/best.pt")
results = model.val()
print(result.box.maps)